In [2]:

# ============================================
# HEART DISEASE PREDICTION - MODEL TRAINING
# CLEAN, CORRECT, DEPLOYMENT-READY VERSION
# ============================================

import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# -------------------------------------------------
# 1. LOAD DATA
# -------------------------------------------------
data = pd.read_csv("dataset.csv")


# -------------------------------------------------
# 2. RENAME COLUMNS (match Streamlit app)
# -------------------------------------------------
data.rename(columns={
    'age': 'Age',
    'sex': 'Gender',
    'cp': 'ChestPainType',
    'trestbps': 'RestingBP',
    'chol': 'Cholesterol',
    'fbs': 'FastingBS',
    'restecg': 'RestingECG',
    'thalach': 'MaxHR',
    'exang': 'ExerciseAngina',
    'oldpeak': 'ST_Depression',
    'slope': 'ST_Slope',
    'ca': 'MajorVessels',
    'thal': 'Thalassemia',
    'target': 'HeartDisease'
}, inplace=True)

# -------------------------------------------------
# 3. SPLIT FEATURES & TARGET
# -------------------------------------------------
X = data.drop("HeartDisease", axis=1)
y = data["HeartDisease"]

# -------------------------------------------------
# 4. DEFINE CATEGORICAL & NUMERIC COLUMNS
# -------------------------------------------------
categorical_cols = [
    'Gender','ChestPainType','FastingBS','RestingECG',
    'ExerciseAngina','ST_Slope','MajorVessels','Thalassemia'
]

numerical_cols = [
    'Age','Cholesterol','RestingBP','MaxHR','ST_Depression'
]

# -------------------------------------------------
# 5. ONE-HOT ENCODING (Correctly only on X)
# -------------------------------------------------
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

# -------------------------------------------------
# 6. SCALE NUMERICAL FEATURES
# -------------------------------------------------
scaler = StandardScaler()
X_encoded[numerical_cols] = scaler.fit_transform(X_encoded[numerical_cols])

print("Final Training Feature Shape:", X_encoded.shape)

# -------------------------------------------------
# 7. TRAIN-TEST SPLIT
# -------------------------------------------------
x_train, x_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

# -------------------------------------------------
# 8. TRAIN RANDOM FOREST CLASSIFIER
# -------------------------------------------------
model = RandomForestClassifier(
    n_estimators=500,
    max_depth=8,
    min_samples_split=5,
    class_weight="balanced",
    random_state=42)
model.fit(x_train, y_train)

# -------------------------------------------------
# 9. MODEL EVALUATION
# -------------------------------------------------
y_pred = model.predict(x_test)
accuracy = accuracy_score(y_test, y_pred)

print("\nModel Accuracy:", round(accuracy * 100, 2), "%")
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n",confusion_matrix(y_test, y_pred))
print("\n",X_encoded.columns)

# -------------------------------------------------
# 10. SAVE MODEL + SCALER + FEATURE NAMES
# -------------------------------------------------
with open("heart-disease-model.pkl", "wb") as file:
    pickle.dump((model, scaler, X_encoded.columns), file)

print("\nModel saved successfully as 'heart-disease-model.pkl'")


Final Training Feature Shape: (303, 22)

Model Accuracy: 77.05 %

Classification Report:
               precision    recall  f1-score   support

           0       0.79      0.68      0.73        28
           1       0.76      0.85      0.80        33

    accuracy                           0.77        61
   macro avg       0.77      0.76      0.77        61
weighted avg       0.77      0.77      0.77        61


Confusion Matrix:
 [[19  9]
 [ 5 28]]

 Index(['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'ST_Depression', 'Gender_1',
       'ChestPainType_1', 'ChestPainType_2', 'ChestPainType_3', 'FastingBS_1',
       'RestingECG_1', 'RestingECG_2', 'ExerciseAngina_1', 'ST_Slope_1',
       'ST_Slope_2', 'MajorVessels_1', 'MajorVessels_2', 'MajorVessels_3',
       'MajorVessels_4', 'Thalassemia_1', 'Thalassemia_2', 'Thalassemia_3'],
      dtype='object')

Model saved successfully as 'heart-disease-model.pkl'


In [3]:
data.info()
data["Gender"].unique()
data.groupby("Gender").mean()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Age             303 non-null    int64  
 1   Gender          303 non-null    int64  
 2   ChestPainType   303 non-null    int64  
 3   RestingBP       303 non-null    int64  
 4   Cholesterol     303 non-null    int64  
 5   FastingBS       303 non-null    int64  
 6   RestingECG      303 non-null    int64  
 7   MaxHR           303 non-null    int64  
 8   ExerciseAngina  303 non-null    int64  
 9   ST_Depression   303 non-null    float64
 10  ST_Slope        303 non-null    int64  
 11  MajorVessels    303 non-null    int64  
 12  Thalassemia     303 non-null    int64  
 13  HeartDisease    303 non-null    int64  
dtypes: float64(1), int64(13)
memory usage: 33.3 KB


,Age,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,ST_Depression,ST_Slope,MajorVessels,Thalassemia,HeartDisease
Gender,,,,,,,,,,,,,
0,55.677083,1.041667,133.083333,261.302083,0.12500,0.572917,151.125000,0.229167,0.876042,1.427083,0.552083,2.125000,0.750000
1,53.758454,0.932367,130.946860,239.289855,0.15942,0.507246,148.961353,0.371981,1.115459,1.386473,0.811594,2.400966,0.449275


In [4]:
print("Gender:", data['Gender'].unique())
print("ChestPainType:", data['ChestPainType'].unique())
print("FastingBS:", data['FastingBS'].unique())
print("RestingECG:", data['RestingECG'].unique())
print("ExerciseAngina:", data['ExerciseAngina'].unique())
print("ST_Slope:", data['ST_Slope'].unique())
print("MajorVessels:", data['MajorVessels'].unique())
print("Thalassemia:", data['Thalassemia'].unique())

Gender: [1 0]
ChestPainType: [3 2 1 0]
FastingBS: [1 0]
RestingECG: [0 1 2]
ExerciseAngina: [0 1]
ST_Slope: [0 2 1]
MajorVessels: [0 2 1 3 4]
Thalassemia: [1 2 3 0]
